# 🛡️ W4: 안전점검 15항목 + Gemini AI
**hexa-5 — 건설헙 AX 마스터클래스 — Week 4**

---
**학습 목표**
1. 건설 현장 안전점검 15개 핵심 항목을 자동으로 체크한다
2. Gemini AI를 활용하여 안전 위험 요인을 분석하고 개선책을 제안한다
3. 안전점검 결과를 시각화하고 보고서를 자동 생성한다

> ⏱️ 예상 소요시간: 60분 | 🎯 결과물: 안전점검 보고서 + AI 개선 제안 (PDF 다운로드)

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib seaborn reportlab
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
from IPython.display import HTML, display
import base64
from io import BytesIO

_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
# === 2026년 3월 기준 최신 모델 선택 ===
# ✅ GA(안정): gemini-2.5-flash  ← 기본값 (권장)
# 🔵 Preview : gemini-3.1-flash-lite-preview  (최신, 2026.3 출시)
# 🔵 Preview : gemini-3.1-pro-preview  (최강, 복잡한 분석용)
# ⚠️  구버전 gemini-2.0-flash 는 2026년 6월 1일 종료 예정
MODEL_NAME = 'gemini-2.5-flash'  # 원하는 모델로 변경하세요
model=genai.GenerativeModel('gemini-2.5-flash'); print('✅ 준비 완료')

## Step 1: 안전점검 15핵심항목 정의

In [ ]:
# 건설 현장 안전점검 15핵심항목
SAFETY_CHECKLIST = [
    {
        'category': '구조물 안전',
        'items': [
            {'id': 'S01', 'name': '가설구조물 안전', 'weight': 10, 'description': '비계, 타워크레인 등 가설구조물의 안전성 확인'},
            {'id': 'S02', 'name': '구조물 안전성', 'weight': 8, 'description': '건축물 구조의 안정성 및 균열 상태 점검'}
        ]
    },
    {
        'category': '전기 안전',
        'items': [
            {'id': 'S03', 'name': '전기설비 접지', 'weight': 9, 'description': '전기기계기의 접지 및 누전차단기 작동 확인'},
            {'id': 'S04', 'name': '전선 관리', 'weight': 7, 'description': '전선 피복 상태 및 적정 관리 상태 점검'}
        ]
    },
    {
        'category': '화재 예방',
        'items': [
            {'id': 'S05', 'name': '소화기 비치', 'weight': 8, 'description': '소화기 설치 및 유효기간 확인'},
            {'id': 'S06', 'name': '인화성 물질 관리', 'weight': 7, 'description': '화약류, 유류 등 인화성 물질의 안전한 보관 확인'}
        ]
    },
    {
        'category': '작업 환경',
        'items': [
            {'id': 'S07', 'name': '통행로 확보', 'weight': 6, 'description': '작업장 통행로의 안전성 및 적정 폭 확보'},
            {'id': 'S08', 'name': '조명 환경', 'weight': 5, 'description': '작업장 조명 상태 및 어두운 구간 점검'},
            {'id': 'S09', 'name': '환기 시스템', 'weight': 6, 'description': '밀폐공간 환기장치 작동 상태 확인'}
        ]
    },
    {
        'category': '장비 안전',
        'items': [
            {'id': 'S10', 'name': '기계장비 점검', 'weight': 8, 'description': '굴삭기, 크레인 등 중장비의 정기점검 상태 확인'},
            {'id': 'S11', 'name': '방호장치', 'weight': 7, 'description': '기계장비 안전방호장치 작동 상태 점검'}
        ]
    },
    {
        'category': '개인 보호구',
        'items': [
            {'id': 'S12', 'name': '안전모 착용', 'weight': 9, 'description': '작업자의 안전모 착용 여부 및 적합성 확인'},
            {'id': 'S13', 'name': '보호구 비치', 'weight': 8, 'description': '안전화, 안전장갑 등 개인보호구 비치 상태'}
        ]
    },
    {
        'category': '비상 조치',
        'items': [
            {'id': 'S14', 'name': '비상구 확보', 'weight': 10, 'description': '비상구의 개방 상태 및 접근성 확인'},
            {'id': 'S15', 'name': '구조장비', 'weight': 8, 'description': '구조용 장비의 비치 및 작동 상태 점검'}
        ]
    }
]

# 체크리스트를 DataFrame으로 변환
checklist_items = []
for category in SAFETY_CHECKLIST:
    for item in category['items']:
        item['category'] = category['category']
        checklist_items.append(item)

df_checklist = pd.DataFrame(checklist_items)
print('📋 안전점검 15핵심항목:')
display(df_checklist[['id', 'name', 'category', 'weight', 'description']])

## Step 2: 안전점검 데이터 생성 함수

In [ ]:
def generate_safety_inspection_data():
    """안전점검 시뮬레이션 데이터 생성"""
    
    # 현장 정보
    sites = [
        {'site_id': 'SITE001', 'name': '송파 오피스빌딩 공사현장', 'location': '서울시 송파구', 'size': '대형'},
        {'site_id': 'SITE002', 'name': '강남 아파트 리모델링', 'location': '서울시 강남구', 'size': '중형'},
        {'site_id': 'SITE003', 'name': '판교 테크밸리 공장', 'location': '경기도 성남시', 'size': '대형'}
    ]
    
    inspection_data = []
    
    for site in sites:
        # 각 현장별 안전점검 결과 생성
        for _, item in df_checklist.iterrows():
            # 안전점검 결과 (0: 불합격, 1: 합격, 2: 우수)
            # 중요도(weight)가 높을수록 합격률이 낮도록 설정
            base_prob = 0.8 - (item['weight'] - 5) * 0.03
            
            if np.random.random() < base_prob:
                score = np.random.choice([1, 2], p=[0.7, 0.3])  # 합격 또는 우수
            else:
                score = 0  # 불합격
            
            inspection_data.append({
                'site_id': site['site_id'],
                'site_name': site['name'],
                'location': site['location'],
                'site_size': site['size'],
                'inspection_date': datetime.now() - timedelta(days=np.random.randint(0, 7)),
                'check_id': item['id'],
                'check_name': item['name'],
                'category': item['category'],
                'weight': item['weight'],
                'score': score,
                'score_text': ['불합격', '합격', '우수'][score],
                'inspector': f'안전관리자{np.random.randint(1, 4)}',
                'notes': generate_notes(score, item['name'])
            })
    
    return pd.DataFrame(inspection_data)

def generate_notes(score, check_name):
    """점검 결과에 따른 비고 생성"""
    notes_templates = {
        0: [  # 불합격
            f'{check_name} 관련 심각한 위험 요인 발견',
            f'{check_name} 관련 즉각적인 조치 필요',
            f'{check_name} 관련 안전기준 미준수',
            f'{check_name} 관련 개선이 시급함'
        ],
        1: [  # 합격
            f'{check_name} 관련 기준 충족',
            f'{check_name} 관련 특이사항 없음',
            f'{check_name} 관련 양호한 상태 유지'
        ],
        2: [  # 우수
            f'{check_name} 관련 모범적인 관리 상태',
            f'{check_name} 관련 우수 사례로 적극 권장',
            f'{check_name} 관련 타 현장 벤치마킹 필요'
        ]
    }
    
    return np.random.choice(notes_templates[score])

# 안전점검 데이터 생성
df_inspection = generate_safety_inspection_data()
print(f'🔍 안전점검 데이터 생성 완료 (총 {len(df_inspection)}건)')
display(df_inspection.head())

## Step 3: 안전점검 결과 분석

In [ ]:
def analyze_safety_results(df):
    """안전점검 결과 종합 분석"""
    
    # 1. 전체 안전점수 계산
    df['weighted_score'] = df['score'] * df['weight']
    
    # 2. 현장별 분석
    site_analysis = df.groupby(['site_id', 'site_name', 'location']).agg({
        'weighted_score': 'sum',
        'weight': 'sum',
        'score': 'mean',
        'check_id': 'count'
    }).rename(columns={
        'check_id': 'total_checks'
    })
    
    site_analysis['safety_rate'] = (site_analysis['weighted_score'] / (site_analysis['weight'] * 2)) * 100  # 최대 점수가 2
    site_analysis['grade'] = site_analysis['safety_rate'].apply(
        lambda x: 'A' if x >= 90 else 'B' if x >= 80 else 'C' if x >= 70 else 'D'
    )
    
    # 3. 카테고리별 분석
    category_analysis = df.groupby('category').agg({
        'weighted_score': 'sum',
        'weight': 'sum',
        'score': 'mean',
        'check_id': 'count'
    }).rename(columns={
        'check_id': 'total_checks'
    })
    
    category_analysis['safety_rate'] = (category_analysis['weighted_score'] / (category_analysis['weight'] * 2)) * 100
    
    # 4. 위험 항목 식별
    failed_items = df[df['score'] == 0].sort_values('weight', ascending=False)
    
    # 5. 통계 요약
    stats = {
        'total_inspections': len(df),
        'failed_checks': len(df[df['score'] == 0]),
        'passed_checks': len(df[df['score'] >= 1]),
        'excellent_checks': len(df[df['score'] == 2]),
        'overall_safety_rate': (df['weighted_score'].sum() / (df['weight'].sum() * 2)) * 100
    }
    
    return site_analysis, category_analysis, failed_items, stats

# 안전점검 결과 분석
site_analysis, category_analysis, failed_items, stats = analyze_safety_results(df_inspection)

print('📊 안전점검 종합 분석 결과:')
print(f'• 총 점검항목: {stats["total_inspections"]}건')
print(f'• 불합격 항목: {stats["failed_checks"]}건')
print(f'• 합격 이상: {stats["passed_checks"]}건')
print(f'• 우수 평가: {stats["excellent_checks"]}건')
print(f'• 전체 안전률: {stats["overall_safety_rate"]:.1f}%')

print('\n🏢 현장별 안전등급:')
display(site_analysis[['safety_rate', 'grade']].round(1))

## Step 4: 안전점검 결과 시각화

In [ ]:
def visualize_safety_results(site_analysis, category_analysis, df):
    """안전점검 결과 시각화"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. 현장별 안전률
    bars1 = axes[0,0].bar(site_analysis['site_name'], site_analysis['safety_rate'], 
                         color=['#FF6B6B', '#4ECDC4', '#45B7D1'], alpha=0.8)
    axes[0,0].set_title('현장별 안전점검 결과', fontsize=14, fontweight='bold')
    axes[0,0].set_ylabel('안전률 (%)', fontsize=12)
    axes[0,0].tick_params(axis='x', rotation=15)
    axes[0,0].grid(True, alpha=0.3)
    
    # 막대에 등급 표시
    for i, bar in enumerate(bars1):
        height = bar.get_height()
        grade = site_analysis.iloc[i]['grade']
        axes[0,0].text(bar.get_x() + bar.get_width()/2., height + 1,
                     f'{height:.1f}% ({grade}등급)', ha='center', va='bottom', fontweight='bold')
    
    # 2. 카테고리별 안전률
    bars2 = axes[0,1].barh(category_analysis.index, category_analysis['safety_rate'], 
                         color='skyblue', alpha=0.8)
    axes[0,1].set_title('카테고리별 안전률', fontsize=14, fontweight='bold')
    axes[0,1].set_xlabel('안전률 (%)', fontsize=12)
    axes[0,1].grid(True, alpha=0.3)
    
    # 막대에 수치 표시
    for i, bar in enumerate(bars2):
        width = bar.get_width()
        axes[0,1].text(width + 1, bar.get_y() + bar.get_height()/2.,
                     f'{width:.1f}%', ha='left', va='center')
    
    # 3. 점수 분포 (파이차트)
    score_counts = df['score_text'].value_counts()
    colors = ['#FF6B6B', '#FFA500', '#32CD32']  # 불합격, 합격, 우수
    wedges, texts, autotexts = axes[1,0].pie(score_counts.values, labels=score_counts.index, 
                                            autopct='%1.1f%%', colors=colors, 
                                            startangle=90, textprops={'fontsize': 12})
    axes[1,0].set_title('점검결과 분포', fontsize=14, fontweight='bold')
    
    # 4. 위험 항목 순위 (상위 10개)
    failed_by_item = df[df['score'] == 0].groupby('check_name').size().sort_values(ascending=False).head(10)
    if not failed_by_item.empty:
        bars4 = axes[1,1].barh(range(len(failed_by_item)), failed_by_item.values, color='red', alpha=0.7)
        axes[1,1].set_yticks(range(len(failed_by_item)))
        axes[1,1].set_yticklabels(failed_by_item.index, fontsize=10)
        axes[1,1].set_xlabel('불합격 횟수', fontsize=12)
        axes[1,1].set_title('불합격 항목 TOP 10', fontsize=14, fontweight='bold')
        axes[1,1].grid(True, alpha=0.3)
        
        # 막대에 수치 표시
        for i, bar in enumerate(bars4):
            width = bar.get_width()
            axes[1,1].text(width + 0.1, bar.get_y() + bar.get_height()/2.,
                         str(int(width)), ha='left', va='center')
    else:
        axes[1,1].text(0.5, 0.5, '불합격 항목 없음', ha='center', va='center', 
                       transform=axes[1,1].transAxes, fontsize=14, color='green')
        axes[1,1].set_title('불합격 항목 TOP 10', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# 시각화 실행
visualize_safety_results(site_analysis, category_analysis, df_inspection)

## Step 5: Gemini AI 안전 리스크 분석

In [ ]:
def analyze_safety_with_gemini(df_inspection, failed_items, stats):
    """Gemini AI로 안전 리스크 심층 분석"""
    
    # 상위 위험 항목 추출
    top_risks = failed_items.head(5) if not failed_items.empty else pd.DataFrame()
    
    # 현장별 위험 현황
    site_risks = df_inspection[df_inspection['score'] == 0].groupby(['site_name', 'category']).size().reset_index(name='risk_count')
    
    prompt = f"""
당신은 건설 안전 전문가입니다. 다음 안전점검 결과를 바탕으로 심층 분석과 개선 제안을 작성해주세요.

## 전체 현황
- 총 점검항목: {stats['total_inspections']}건
- 불합격 항목: {stats['failed_checks']}건
- 전체 안전률: {stats['overall_safety_rate']:.1f}%

## 주요 위험 항목 (상위 5개)
{top_risks[['site_name', 'check_name', 'category', 'notes']].to_string(index=False) if not top_risks.empty else '불합격 항목 없음'}

## 현장별 위험 현황
{site_risks.to_string(index=False) if not site_risks.empty else '위험 현황 없음'}

## 분석 요구사항
1. **위험 요인 분석**: 불합격 항목의 근본 원인 분석
2. **연쇄 위험**: 특정 안전 문제가 다른 영역에 미치는 영향
3. **단기 대책**: 즉시 시행 가능한 안전 조치 (구체적이고 실용적인 방안)
4. **장기 개선**: 재발 방지를 위한 시스템적 개선 방안
5. **교육 필요성**: 안전 교육의 우선순위와 교육 내용 제안
6. **법적 측면**: 산업안전보건법 관련 규정 및 준수 방안

## 출력 형식
- 마크다운 형식
- 명확한 헤더 구조
- 실무자가 바로 적용할 수 있는 구체적 내용
- 위험도에 따른 우선순위 제시

건설 현장 실무 경험을 바탕으로 실질적인 분석과 제안을 부탁드립니다.
"""
    
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"AI 분석 중 오류 발생: {e}")
        return "AI 분석을 통한 안전 리스크 평가를 생성할 수 없습니다."

# AI 안전 리스크 분석 실행
print('🤖 Gemini AI 안전 리스크 심층 분석:')
print('=' * 60)
safety_analysis = analyze_safety_with_gemini(df_inspection, failed_items, stats)
print(safety_analysis)

## Step 6: 안전점검 보고서 생성

In [ ]:
def generate_safety_report(site_analysis, category_analysis, failed_items, stats, safety_analysis):
    """안전점검 종합 보고서 생성"""
    
    report_date = datetime.now().strftime('%Y년 %m월 %d일')
    
    report = f"""
# 건설 현장 안전점검 종합 보고서

> 보고일: {report_date}
> 점검 대상: {len(site_analysis)}개 현장

## 1. 개요

본 보고서는 건설 현장 안전점검 15핵심항목에 대한 종합 분석 결과입니다.
각 현장의 안전 수준을 평가하고 개선이 필요한 사항을 도출하였습니다.

## 2. 종합 평가

| 구분 | 결과 | 평가 |
|------|------|------|
| 총 점검항목 | {stats['total_inspections']}건 | - |
| 불합격 항목 | {stats['failed_checks']}건 | {'위험' if stats['failed_checks'] > 5 else '주의' if stats['failed_checks'] > 2 else '양호'} |
| 합격률 | {stats['passed_checks']/stats['total_inspections']*100:.1f}% | {'우수' if stats['passed_checks']/stats['total_inspections'] >= 0.9 else '양호' if stats['passed_checks']/stats['total_inspections'] >= 0.8 else '개선필요'} |
| 전체 안전률 | {stats['overall_safety_rate']:.1f}% | {'매우우수' if stats['overall_safety_rate'] >= 95 else '우수' if stats['overall_safety_rate'] >= 90 else '양호' if stats['overall_safety_rate'] >= 80 else '개선필요'} |

## 3. 현장별 평가 결과

{site_analysis[['safety_rate', 'grade']].to_string(float_format='%.1f')}

### 현장별 특징
"""
    
    # 현장별 특징 추가
    for site_id, site_data in site_analysis.iterrows():
        site_name = site_data.name[1]  # MultiIndex에서 site_name 추출
        grade = site_data['grade']
        safety_rate = site_data['safety_rate']
        
        if grade == 'A':
            eval_text = "모범적인 안전관리 상태이며 타 현장의 벤치마킹 사례로 적합함"
        elif grade == 'B':
            eval_text = "안전관리 수준은 양호하나 일부 개선이 필요함"
        elif grade == 'C':
            eval_text = "안전관리에 시급한 개선이 필요하며 집중적인 관리가 요구됨"
        else:
            eval_text = "위험 수준이 높으며 즉각적인 안전 조치가 시급함"
        
        report += f"""
- **{site_name}**: {safety_rate:.1f}% ({grade}등급) - {eval_text}
"""
    
    report += f"""

## 4. 카테고리별 분석

{category_analysis[['safety_rate']].to_string(float_format='%.1f')}

## 5. 주요 위험 항목

"""
    
    if not failed_items.empty:
        report += failed_items[['site_name', 'check_name', 'category', 'notes']].head(10).to_string(index=False)
    else:
        report += "\n위험 항목이 없어 안전 관리 상태가 양호합니다."
    
    report += f"""

## 6. AI 기반 안전 리스크 분석 및 제언

{safety_analysis}

## 7. 개선 과제 및 후속 조치

### 7.1 단기 개선 과제 (1개월 이내)
1. 불합격 항목에 대한 즉각적인 시정 조치
2. 안전장비 보강 및 정비
3. 근로자 안전교육 강화

### 7.2 중기 개선 과제 (3개월 이내)
1. 안전관리 프로세스 표준화
2. 정기 안전점검 시스템 구축
3. 안전성과 평가제도 도입

### 7.3 장기 개선 과제 (6개월 이내)
1. 스마트 안전관리 시스템 구축
2. 안전문화 조성 및 확산
3. 산업안전보건법 규정 완전 준수

## 8. 결론

전반적인 안전 관리 수준은 {'우수하나' if stats['overall_safety_rate'] >= 90 else '양호하나' if stats['overall_safety_rate'] >= 80 else '보통이나'} 일부 개선이 필요합니다.
특히 {stats['failed_checks']}개의 불합격 항목에 대한 집중적인 관리가 시급합니다.

본 보고서의 제언사항을 적극 반영하여 재해율 제로 목표를 달성해야 합니다.

---
*본 보고서는 AI 자동 분석 시스템을 통해 생성되었습니다.*
"""
    
    return report

# 안전점검 보고서 생성
safety_report = generate_safety_report(site_analysis, category_analysis, failed_items, stats, safety_analysis)

# 보고서 저장
with open('건설_안전점검_종합_보고서.md', 'w', encoding='utf-8') as f:
    f.write(safety_report)

print('📋 안전점검 종합 보고서 생성 완료!')
print('📁 파일명: 건설_안전점검_종합_보고서.md')
print('\n📄 보고서 상단 500자 미리보기:')
print(safety_report[:500] + '...')

## Step 7: 안전점검 결과 저장 및 다운로드

In [ ]:
import os
import zipfile

# 저장할 디렉토리 생성
!mkdir -p safety_inspection_results

# 1. 상세 점검 결과 저장
detailed_results = df_inspection.copy()
detailed_results['inspection_date'] = detailed_results['inspection_date'].dt.strftime('%Y-%m-%d %H:%M')
detailed_results.to_excel('safety_inspection_results/상세_안전점검_결과.xlsx', index=False, encoding='utf-8-sig')

# 2. 요약 통계 저장
summary_stats = {
    '전체 안전률': [stats['overall_safety_rate']],
    '불합격률': [stats['failed_checks']/stats['total_inspections']*100],
    '우수 평가율': [stats['excellent_checks']/stats['total_inspections']*100],
    '점검 현장 수': [len(site_analysis)],
    '총 점검 항목': [stats['total_inspections']],
    '보고 생성일': [datetime.now().strftime('%Y-%m-%d')]
}

pd.DataFrame(summary_stats).to_excel('safety_inspection_results/요약_통계.xlsx', index=False, encoding='utf-8-sig')

# 3. 위험 항목 목록 저장
if not failed_items.empty:
    failed_items[['site_name', 'check_name', 'category', 'weight', 'notes']].to_excel(
        'safety_inspection_results/위험_항목_목록.xlsx', index=False, encoding='utf-8-sig'
    )

# 4. 현장별 평가 결과 저장
site_results = site_analysis.reset_index()
site_results.to_excel('safety_inspection_results/현장별_평가.xlsx', index=False, encoding='utf-8-sig')

# 5. 카테고리별 분석 결과 저장
category_results = category_analysis.reset_index()
category_results.to_excel('safety_inspection_results/카테고리별_분석.xlsx', index=False, encoding='utf-8-sig')

# ZIP 파일 생성
with zipfile.ZipFile('건설_안전점검_결과_모음.zip', 'w') as zipf:
    for foldername, subfolders, filenames in os.walk('safety_inspection_results'):
        for filename in filenames:
            filepath = os.path.join(foldername, filename)
            zipf.write(filepath, filename)
    
    # 보고서 파일도 추가
    zipf.write('건설_안전점검_종합_보고서.md', '건설_안전점검_종합_보고서.md')

# 생성된 파일 목록
print('✅ 모든 안전점검 결과 저장 완료!')
print('\n📁 생성된 파일 목록:')
for filename in os.listdir('safety_inspection_results'):
    print(f'  - {filename}')

# 다운로드
try:
    from google.colab import files
    files.download('건설_안전점검_결과_모음.zip')
    files.download('건설_안전점검_종합_보고서.md')
    print('\n🎉 파일 다운로드 완료!')
except:
    print('\n⚠️ 다운로드 기능을 사용할 수 없습니다.')

# 최종 요약
print('\n🎯 최종 요약:')
print(f'• 총 {len(site_analysis)}개 현장 안전점검 완료')
print(f'• 전체 안전률: {stats["overall_safety_rate"]:.1f}%')
print(f'• 개선 필요 항목: {stats["failed_checks"]}건')
print('• AI 기반 위험 분석 및 개선 제안 완료')